<a href="https://colab.research.google.com/github/Fa-Alsuqayri/Agentic-AI-System-Assignments/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Faisal Alsuqayri

---

##Assignment 4

###Use case: BERT paper

In [ ]:
%pip install langchain-community pypdf
%pip install langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore


In [ ]:
!curl -L -o bert_paper.pdf "https://arxiv.org/pdf/1810.04805.pdf"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   217  100   217    0     0   1706      0 --:--:-- --:--:-- --:--:--  1708
100  756k  100  756k    0     0  2774k      0 --:--:-- --:--:-- --:--:-- 2774k


In [ ]:

file_path = "bert_paper.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(len(docs))

16


In [ ]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout

{'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'bert_paper.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

83


In [1]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.0052108620293438435, 0.027721881866455078, 0.001389949582517147, 0.08294462412595749, 0.001529207220301032, 0.025097651407122612, 0.008785158395767212, -0.04010576382279396, -0.02537068910896778, -0.051830556243658066]


In [ ]:
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
results = vector_store.similarity_search(
    "What does BERT stand for?"
)

print(results[0])

page_content='BERT including:
– Effect of Number of Training Steps; and
– Ablation for Different Masking Proce-
dures.
A Additional Details for BERT
A.1 Illustration of the Pre-training Tasks
We provide examples of the pre-training tasks in
the following.
Masked LM and the Masking Procedure As-
suming the unlabeled sentence is my dog is
hairy, and during the random masking procedure
we chose the 4-th token (which corresponding to
hairy), our masking procedure can be further il-
lustrated by
• 80% of the time: Replace the word with the
[MASK] token, e.g., my dog is hairy →
my dog is [MASK]
• 10% of the time: Replace the word with a
random word, e.g., my dog is hairy → my
dog is apple
• 10% of the time: Keep the word un-
changed, e.g., my dog is hairy → my dog
is hairy . The purpose of this is to bias the
representation towards the actual observed
word.
The advantage of this procedure is that the
Transformer encoder does not know which words
it will be asked to predict or which have been

In [ ]:
results = await vector_store.asimilarity_search("How is BERT different from OpenAI GPT?")

print(results[0])

page_content='size as OpenAI GPT for comparison purposes.
Critically, however, the BERT Transformer uses
bidirectional self-attention, while the GPT Trans-
former uses constrained self-attention where every
token can only attend to context to its left.4
1https://github.com/tensorﬂow/tensor2tensor
2http://nlp.seas.harvard.edu/2018/04/03/attention.html
3In all cases we set the feed-forward/ﬁlter size to be 4H,
i.e., 3072 for the H = 768 and 4096 for the H = 1024.
4We note that in the literature the bidirectional Trans-' metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'bert_paper.pdf', 'total_pages': 16, 'page': 2, 'page_label': '3', 'start_index': 3195}


In [ ]:
results = vector_store.similarity_search_with_score("What are the two unsupervised tasks used for pre-training BERT?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.7083076121247822

page_content='BERT including:
– Effect of Number of Training Steps; and
– Ablation for Different Masking Proce-
dures.
A Additional Details for BERT
A.1 Illustration of the Pre-training Tasks
We provide examples of the pre-training tasks in
the following.
Masked LM and the Masking Procedure As-
suming the unlabeled sentence is my dog is
hairy, and during the random masking procedure
we chose the 4-th token (which corresponding to
hairy), our masking procedure can be further il-
lustrated by
• 80% of the time: Replace the word with the
[MASK] token, e.g., my dog is hairy →
my dog is [MASK]
• 10% of the time: Replace the word with a
random word, e.g., my dog is hairy → my
dog is apple
• 10% of the time: Keep the word un-
changed, e.g., my dog is hairy → my dog
is hairy . The purpose of this is to bias the
representation towards the actual observed
word.
The advantage of this procedure is that the
Transformer encoder does not know which words
it will be asked to